#### Gradio 기본 인터페이스와 챗봇 만들기
* 각 예제는 **셀 단위로 독립 실행**됩니다. 새 예제를 실행하면 이전 서버가 자동으로 종료되어 포트(7860)가 해제됩니다.
* 패턴: `build_xxx()`에서 UI를 **정의**하고, `launch_demo(build_xxx())`로 **실행**합니다.

##### 0. 공통 헬퍼: 셀별 실행과 포트 해제
* `launch_demo(demo)`: 기존 서버를 먼저 닫고 새로 실행합니다. (`prevent_thread_lock=True`라서 셀이 블로킹되지 않습니다)
* `close_demo()`: 실행 중인 Gradio 서버를 모두 종료해 포트를 해제합니다.
* `is_port_open()`: 포트 점유 여부를 확인합니다.

In [ ]:
import socket
import gradio as gr

GRADIO_HOST = "127.0.0.1"   # 로컬에서만 접속 가능
GRADIO_PORT = 7860          # 모든 예제가 공유하는 포트

def is_port_open(port: int = GRADIO_PORT, host: str = GRADIO_HOST) -> bool:
    """해당 포트에서 서버가 응답하면 True(점유 중), 아니면 False(해제됨)를 반환한다."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0

def close_demo() -> None:
    """실행 중인 모든 Gradio 서버를 종료하여 포트를 해제한다."""
    gr.close_all()

def launch_demo(demo: gr.Blocks, port: int = GRADIO_PORT) -> None:
    """이전 서버를 종료한 뒤 demo를 실행한다. 셀을 다시 실행해도 포트 충돌이 없다."""
    close_demo()
    demo.launch(server_name=GRADIO_HOST, server_port=port, prevent_thread_lock=True)

print("Gradio", gr.__version__, "/ 포트 점유 여부:", is_port_open())

##### 1. 가장 간단한 Interface
* `gr.Interface`로 사용자의 이름을 입력받아 "안녕!"을 붙여 반환하는 인터페이스를 만듭니다.
* `inputs`/`outputs`에 `"text"`처럼 문자열 단축형을 쓸 수 있습니다.
* `fn`의 인자는 `inputs`, `fn`의 반환값은 `outputs`에 대응합니다.

In [ ]:
def greet(name: str) -> str:
    """이름 앞에 '안녕! '을 붙여 반환한다. (이후 예제에서도 재사용)"""
    return "안녕! " + name

def build_simple() -> gr.Interface:
    """문자열 단축형(\"text\")으로 입출력을 지정한 Interface를 만든다."""
    return gr.Interface(fn=greet, inputs="text", outputs="text")

launch_demo(build_simple())

##### 2. gr.Textbox로 입력창 꾸미기
* 앞 예제와 동일하게 동작하지만, `gr.Textbox`를 사용해 입력창에 추가 정보를 제공합니다.
    * `lines`: 입력창 높이(줄 수), `placeholder`: 입력창에 보이는 힌트
    * 사용자가 어떤 정보를 입력해야 하는지 더 잘 이해할 수 있습니다.

In [ ]:
def build_textbox() -> gr.Interface:
    """Textbox 옵션(lines, placeholder, label)을 적용한 Interface를 만든다."""
    return gr.Interface(
        fn=greet,
        inputs=gr.Textbox(lines=2, placeholder="이름을 작성하세요.", label="이름"),
        outputs=gr.Textbox(label="인사말"),
    )

launch_demo(build_textbox())

##### 참고. 화면의 Flag 버튼은 무엇인가요?
* `gr.Interface`로 만든 화면(예제 1, 2)에는 **Flag** 버튼이 기본으로 표시됩니다. (`gr.Blocks`로 만든 화면에는 없습니다)
* **용도**: 사용자가 틀린 답이나 이상한 출력을 표시해 두면, 개발자가 나중에 CSV로 실패 사례를 검토·디버깅하거나 평가/학습 데이터로 활용합니다. 챗봇 개발에서는 잘못된 답변을 모아 프롬프트를 개선할 때 유용합니다.
* 클릭하면 입력과 출력이 `.gradio/flagged/dataset1.csv`에 한 줄 저장됩니다. 폴더는 Flag를 **처음 클릭할 때** 만들어집니다.
* 필요 없으면 `gr.Interface(..., flagging_mode="never")`로 끕니다. 저장 폴더는 `flagging_dir`로 바꿀 수 있습니다.
* 입력 내용이 평문으로 저장되므로 개인정보나 API 키 입력에 주의하고, `.gradio/` 폴더는 git에 커밋하지 마세요.

##### 3. gr.Blocks와 Button 이벤트
* `gr.Blocks()`로 컴포넌트를 직접 배치하고, 이벤트 리스너로 함수와 연결합니다.
* `버튼.click(fn, inputs, outputs)`: 버튼을 클릭할 때마다 `greet`가 호출되어 출력창에 결과를 표시합니다.
* `api_name`을 지정하면 이 이벤트를 API 엔드포인트(`/greet`)로 호출할 수 있습니다.

In [ ]:
def build_blocks() -> gr.Blocks:
    """Textbox 2개와 Button을 Blocks로 배치하고 click 이벤트를 연결한다."""
    with gr.Blocks() as demo:
        name = gr.Textbox(label="이름")
        output = gr.Textbox(label="출력창")
        greet_btn = gr.Button("인사")
        greet_btn.click(fn=greet, inputs=name, outputs=output, api_name="greet")
    return demo

launch_demo(build_blocks())

##### 4. gr.Chatbot으로 간단한 채팅봇 만들기
* 사용자의 메시지를 입력받고, 미리 정의된 응답 중 하나를 무작위로 선택해 반환합니다.
* 대화 기록(`chat_history`)은 `{"role": ..., "content": ...}` 딕셔너리 리스트(messages 형식)입니다.
* 입력창에서 Enter를 누르면(`submit`) 응답하고, 입력창은 `""`로 비웁니다.
* '초기화' 버튼은 기록을 빈 리스트로 되돌립니다.

In [ ]:
import random

BOT_REPLIES = ["어떻게 지내세요?", "좋아해요", "배고파요"]   # 무작위 응답 후보

def respond(message: str, chat_history: list):
    """사용자 메시지와 봇 응답을 기록에 추가하고, 입력창을 비운다."""
    if not message.strip():                 # 빈 입력은 무시
        return "", chat_history
    bot_message = random.choice(BOT_REPLIES)
    chat_history = chat_history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": bot_message},
    ]
    return "", chat_history

def build_chatbot() -> gr.Blocks:
    """Chatbot + Textbox + 초기화 Button으로 구성된 채팅 UI를 만든다."""
    with gr.Blocks() as demo:
        chatbot = gr.Chatbot(label="채팅창")
        msg = gr.Textbox(label="입력")
        clear = gr.Button("초기화")

        msg.submit(respond, inputs=[msg, chatbot], outputs=[msg, chatbot])
        clear.click(lambda: [], inputs=None, outputs=chatbot, queue=False)
    return demo

launch_demo(build_chatbot())

##### 5. gr.ChatInterface와 스트리밍 응답
* `gr.ChatInterface`는 4번의 Chatbot + Textbox + 버튼 구성을 한 줄로 만들어 주는 고수준 API입니다. 대화 기록 관리도 자동입니다.
* 함수가 `yield`로 값을 내보내면 응답이 **스트리밍**됩니다. (이전의 `time.sleep(1)` 방식은 UI를 멈추게 만들었습니다)
* 함수 시그니처: `fn(message, history)` — 반환(또는 yield)하는 문자열이 봇의 응답입니다.
* LLM 챗봇을 만들 때는 보통 이 방식을 사용합니다.

In [ ]:
import time

STREAM_DELAY = 0.05   # 글자 사이 지연(초) - 타이핑 효과

def stream_reply(message: str, history: list):
    """무작위 응답을 한 글자씩 yield하여 스트리밍한다."""
    reply = random.choice(BOT_REPLIES)
    partial = ""
    for ch in reply:
        partial += ch
        time.sleep(STREAM_DELAY)
        yield partial

def build_chat_interface() -> gr.ChatInterface:
    """ChatInterface로 스트리밍 챗봇을 만든다."""
    return gr.ChatInterface(
        fn=stream_reply,
        title="간단한 채팅봇",
        description="무작위 응답을 스트리밍으로 보여 줍니다.",
    )

launch_demo(build_chat_interface())

##### 참고. 챗봇의 피드백 버튼
* `gr.ChatInterface`는 기본값이 `flagging_mode="never"`라서 피드백 버튼이 **보이지 않습니다**. `"manual"`로 바꾸면 봇 메시지마다 👍/👎 아이콘이 나타납니다.
* **용도**: 어떤 응답이 좋았고 나빴는지 대화 단위로 모아 두었다가, 모델·프롬프트를 개선하거나 평가 데이터로 사용합니다.
* 저장 위치는 `flagging_dir/log.csv`이며 열은 `conversation`(전체 대화 JSON), `index`(몇 번째 메시지), `value`(Like/Dislike), `flag`, `timestamp`입니다.
* `Interface`와 달리 폴더는 앱을 **만들 때** 바로 생성됩니다.
* `flagging_options=["Like", "Dislike", "부정확"]`처럼 지정하면, `Like`/`Dislike` 외의 항목은 별도 깃발 아이콘 메뉴에 표시됩니다.
* `Blocks`로 직접 만든 `gr.Chatbot`은 `chatbot.like(fn, ...)` 이벤트로 피드백을 받아 원하는 방식으로 저장합니다.

In [ ]:
FEEDBACK_DIR = "flagged_chat"   # 피드백 저장 폴더 (log.csv 생성)

def build_chat_feedback() -> gr.ChatInterface:
    """ 피드백 버튼을 켠 ChatInterface를 만든다. (stream_reply는 앞 셀에서 정의)"""
    return gr.ChatInterface(
        fn=stream_reply,
        title="피드백 버튼이 있는 채팅봇",
        flagging_mode="manual",
        flagging_options=["Like", "Dislike"],
        flagging_dir=FEEDBACK_DIR,
    )

launch_demo(build_chat_feedback())

##### 6. 서버 종료와 포트 해제 확인
* 실습이 끝나면 아래 셀을 실행해 서버를 종료하고 포트가 해제되었는지 확인합니다.

In [ ]:
close_demo()
print("포트 점유 여부:", is_port_open())   # False면 해제 완료